<a href="https://colab.research.google.com/github/alexj11324/TSSPIM-Tsunami-and-Storm-Surge-Population-Impact-Model/blob/migrate/notebooks/shelter_demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Storm Surge Shelter Demand Model — Full E2E Pipeline

**American Red Cross — Storm Surge Shelter Demand Estimation**

This notebook runs the complete pipeline from NHC advisory to census-tract-level shelter demand:

1. **Download** NHC P-Surge raster for the specified storm/advisory
2. **Download** NSI building inventory for affected states
3. **Filter & Map** buildings via DuckDB (bbox clip, dedup, column mapping)
4. **Run FAST** engine to produce building-level damage predictions
5. **Classify** each building into Slight/Moderate/Extensive/Complete damage states
6. **Aggregate** to census tract level and compute BHI (Building Habitability Index)
7. **Join** Census population + CDC SVI data
8. **Estimate** shelter-seeking population (low/high range)

**How to use:**
1. Paste JSON params from Excel Step 6 into Cell 2
2. Run All Cells
3. Copy output from Cell 13 back into Excel Step 7

In [ ]:
import subprocess, sys, os, io, re, json, time, warnings
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'duckdb', 'pyarrow', 'rasterio', 'geopandas',
    'requests', 'openpyxl', 'pygris', 'tqdm', 'utm', 'huggingface_hub'])

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
import utm
from rasterio.warp import transform_bounds
from shapely.geometry import box

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
REPO_URL = 'https://github.com/alexj11324/TSSPIM-Tsunami-and-Storm-Surge-Population-Impact-Model.git'

REPO_BRANCH = 'main' #TODO: Enter branch name here, DEFAULT 'main'
REPO_DIR = Path('repo')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)

WORK_DIR = REPO_DIR.resolve() / "outputs"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Repo root on sys.path so `from scripts.*` works in all later cells (not only after raster download).
repo_root = REPO_DIR.resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Local dev fallback: if running from notebooks/, try cwd or parent for scripts/.
if not (repo_root / "scripts" / "nsi_downloader.py").is_file():
    for _cand in (Path.cwd(), Path.cwd().parent):
        if (_cand / "scripts" / "nsi_downloader.py").is_file():
            _resolved = str(_cand.resolve())
            if _resolved not in sys.path:
                sys.path.insert(0, _resolved)
            break

print(f'Work dir: {WORK_DIR}')
print(f'Environment ready. Branch={REPO_BRANCH}')


Work dir: /content/repo/outputs
Environment ready. Branch=migrate-new-excel


## Config

In [ ]:
# Cell 2: Configuration
# =========================================================
# Reads the Interface sheet of the Excel config file.
# Priority: Google Drive (shared with ARC staff) → repo fallback.

from scripts.read_excel_config import load_config_from_excel

EXCEL_FILENAME = 'ARC Storm Surge Shelter Demand.xlsx'

# Google Drive path (edit this to match your Drive folder)
GDRIVE_EXCEL = Path('/content/drive/MyDrive/Red Cross Capstone Project/Final Report and Deliverables/ARC Storm Surge Shelter Demand.gsheet') #MODIFY THE DIR BEFORE USE

# Repo fallback
REPO_EXCEL = REPO_DIR / 'data' / EXCEL_FILENAME

if GDRIVE_MOUNTED and GDRIVE_EXCEL.exists():
    excel_config_path = GDRIVE_EXCEL
    print(f'Reading config from Google Drive: {excel_config_path}')
else:
    excel_config_path = REPO_EXCEL
    print(f'Reading config from repo: {excel_config_path}')

params = load_config_from_excel(excel_config_path)

# ── Validate ──────────────────────────────────────────────────
for state, ratios in params['BLDNG_USABILITY'].items():
    total = sum(ratios.values())
    if abs(total - 1.0) > 0.01:
        raise ValueError(f'Usability ratios for {state} sum to {total}, expected 1.0')

for sev, classes in params['UL_SEVERITY'].items():
    for cls, bounds in classes.items():
        if not (0 <= bounds[0] <= bounds[1] <= 1):
            raise ValueError(f'UL_SEVERITY[{sev}][{cls}] = {bounds} out of [0,1] range')

if len(params['SVI_SHELTER_RATES']) != 3:
    raise ValueError(f"SVI_SHELTER_RATES must contain exactly 3 values, got {len(params['SVI_SHELTER_RATES'])}")
for v in params['SVI_SHELTER_RATES']:
    if not 0 <= v <= 1:
        raise ValueError(f'SVI_SHELTER_RATES value {v} out of [0,1]')

_svi_bins = params['SVI_BINS']
if not (isinstance(_svi_bins, (list, tuple)) and len(_svi_bins) == 2):
    raise ValueError(f'SVI_BINS must be a list of 2 values, got {_svi_bins!r}')
if not (0 <= _svi_bins[0] < _svi_bins[1] <= 1):
    raise ValueError(f'SVI_BINS must satisfy 0 <= bins[0] < bins[1] <= 1, got {_svi_bins!r}')

if params['flood_load_condition'] not in ('CoastalA', 'CoastalV', 'Riverine'):
    raise ValueError(f"flood_load_condition must be CoastalA/CoastalV/Riverine, got {params['flood_load_condition']}")

for key in ('fast_timeout', 'download_timeout', 'svi_timeout'):
    if params[key] <= 0:
        raise ValueError(f'{key} must be > 0, got {params[key]}')

for key in ('census_max_workers', 'svi_rest_page_size'):
    if not isinstance(params[key], int) or params[key] <= 0:
        raise ValueError(f'{key} must be a positive integer, got {params[key]!r}')

print(f'Storm: {params["storm_name"]} ({params["storm_id"]}), Advisory #{params["advisory"]}, Year {params["year"]}')
print(f'FLC: {params["flood_load_condition"]}, Damage thresholds: {params["DAMAGE_STATE_THRESHOLDS"]}')
print('Params validated OK.')


Storm: BERYL (AL022024), Advisory #29, Year 2024
FLC: CoastalA, Damage thresholds: {'Slight': (0, 15), 'Moderate': (15, 40), 'Extensive': (40, 60), 'Complete': (60, 100)}
Params validated OK.


## Stage 1: Download NHC P-Surge Raster

Downloads the storm surge inundation raster directly from NHC for the specified storm and advisory.
Also identifies which US states are within the raster footprint.

In [ ]:
# Cell 3: Download NHC P-Surge Raster

from scripts.import_nhc_by_storm import download_surge_raster

raster_path, affected_states = download_surge_raster(
    params['storm_id'],
    params['storm_name'],
    params['advisory'],
    params['year'],
    output_dir=WORK_DIR,
    timeout=params['download_timeout'],
)

print(f'Raster saved: {raster_path}')
print(f'Affected states: {affected_states}')

Reading BERYL_2024_adv29_e10_ResultMaskRaster.tif from archive...
Raster saved: /content/repo/outputs/BERYL_2024_adv29_e10_ResultMaskRaster.tif
Affected states: ['Texas', 'Louisiana']


## Stage 2: Download NSI Building Inventory

Downloads NSI (National Structure Inventory) for each affected state.
The `cbfips` field is critical for census tract GEOID derivation.

**Two download sources available** (set `NSI_SOURCE` below):
- `"usace"` (default) — Stream from USACE API (no token needed, slower for large states)
- `"huggingface"` — Download pre-processed Parquet from [Alexq847182/NSI_Parquet](https://huggingface.co/datasets/Alexq847182/NSI_Parquet) (faster, optional HF token for private/gated datasets)

NSI_SOURCE below is a notebook-only switch; CLI scripts default to the USACE API.


In [ ]:
# Cell 4: Download NSI Data
# ========================
# Set NSI_SOURCE to choose download backend:
#   "usace"       — USACE API (default, no token needed)
#   "huggingface" — HuggingFace dataset Alexq847182/NSI_Parquet
NSI_SOURCE = "huggingface"  # <-- change to "huggingface" to use HF

# Optional: set HF token for private/gated datasets (leave None for public)
# or paste your token: "hf_..."

HF_TOKEN = None #TODO


from scripts.nsi_downloader import NSIDownloader

with rasterio.open(raster_path) as src:
    bounds = src.bounds
    raster_bbox_poly = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    if src.crs and str(src.crs) != "EPSG:4326":
        b = transform_bounds(src.crs, "EPSG:4326", *bounds)
        raster_bbox_poly = box(*b)

downloader = NSIDownloader(WORK_DIR)

if NSI_SOURCE == "huggingface":
    nsi_df = downloader.download_states_hf(affected_states, token=HF_TOKEN)
else:
    nsi_df = downloader.download_states(
        affected_states, raster_bbox_polygon=raster_bbox_poly
    )

print(f"\nTotal NSI buildings ({NSI_SOURCE}): {len(nsi_df):,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total: 12,040,947 buildings

Total NSI buildings (huggingface): 12,040,947


In [ ]:
# Cell 5: Spatial Filter + FAST Input Preparation

from scripts.duckdb_fast_pipeline import FOUND_TYPE_MAP, FOUND_TYPE_DEFAULT

bbox = raster_bbox_poly.bounds  # (minx, miny, maxx, maxy)
print(f'Raster bbox (WGS84): lon=[{bbox[0]:.4f}, {bbox[2]:.4f}], lat=[{bbox[1]:.4f}, {bbox[3]:.4f}]')

nsi_filtered = nsi_df[
    nsi_df['latitude'].between(bbox[1], bbox[3]) &
    nsi_df['longitude'].between(bbox[0], bbox[2]) &
    nsi_df['latitude'].notna() &
    nsi_df['bid'].notna()
].copy()

nsi_filtered = (
    nsi_filtered.sort_values('val_struct', ascending=False)
    .drop_duplicates(subset='bid', keep='first')
)

# Normalize cbfips to 15-digit strings to preserve leading zeros in downstream joins
nsi_filtered['cbfips'] = (
    nsi_filtered['cbfips']
    .astype('string')
    .str.replace(r"[^0-9]", '', regex=True)
    .str.zfill(15)
)

fast_input = pd.DataFrame({
    'FltyId': nsi_filtered['bid'],
    'Occ': nsi_filtered['occtype'].str.split('-').str[0].str.upper(),
    'Cost': nsi_filtered['val_struct'].fillna(0),
    'Area': nsi_filtered['sqft'].fillna(0),
    'NumStories': nsi_filtered['num_story'].fillna(1),
    'FoundationType': nsi_filtered['found_type'].astype(str).str.upper().str.strip().map(FOUND_TYPE_MAP).fillna(FOUND_TYPE_DEFAULT).astype(int),
    'FirstFloorHt': nsi_filtered['found_ht'].fillna(1.0),
    'ContentCost': nsi_filtered['val_cont'].fillna(0),
    'Latitude': nsi_filtered['latitude'],
    'Longitude': nsi_filtered['longitude'],
})

fast_csv_path = str(WORK_DIR / 'fast_input.csv')
fast_input.to_csv(fast_csv_path, index=False)
print(f'\nFAST input: {len(fast_input):,} buildings -> {fast_csv_path}')
print(f'  Residential: {fast_input["Occ"].str.startswith("RES").sum():,}')
nsi_cbfips_join_path = str(WORK_DIR / 'nsi_cbfips_join.csv')
nsi_filtered[['bid', 'cbfips']].rename(columns={'bid': 'fltyid'}).to_csv(nsi_cbfips_join_path, index=False)
print(f'  cbfips join table -> {nsi_cbfips_join_path}')
fast_input.head()

Raster bbox (WGS84): lon=[-98.5357, -93.5538], lat=[25.8206, 30.8809]

FAST input: 4,587,280 buildings -> /content/repo/outputs/fast_input.csv
  Residential: 4,101,923
  cbfips join table -> /content/repo/outputs/nsi_cbfips_join.csv


,FltyId,Occ,Cost,Area,NumStories,FoundationType,FirstFloorHt,ContentCost,Latitude,Longitude
6477380,76X6PJ63+2QX-44-31-40-38,IND5,588218744.0,2772321.42,21,7,0.5,882328116.0,29.710122,-95.395504
1869530,76X775VC+GQ8-3-2-3-3,COM7,412493619.0,3010380.62,203,7,0.5,618740428.0,29.293785,-94.828053
6438411,76X6MGHC+494-6-5-9-4,GOV1,402902303.0,2252524.64,419,7,0.5,402902303.0,29.677763,-95.479047
734709,76X7Q555+WPX-9-7-7-7,IND2,302776664.0,2953241.76,308,7,0.5,454164996.0,29.759870,-94.840634
6787437,76X6XMP5+MX9-127-286-114-382,COM8,265796990.0,1614120.00,1,7,0.5,265796990.0,29.986670,-95.340030


In [ ]:
# Cell 6: Run FAST Engine

mapping = {
    'UserDefinedFltyId': 'FltyId', 'OCC': 'Occ', 'Cost': 'Cost',
    'Area': 'Area', 'NumStories': 'NumStories',
    'FoundationType': 'FoundationType', 'FirstFloorHt': 'FirstFloorHt',
    'ContentCost': 'ContentCost', 'Latitude': 'Latitude', 'Longitude': 'Longitude',
}

mapping_path = str(WORK_DIR / 'fast_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(mapping, f)

# --- Pre-flight validation ---
csv_columns = pd.read_csv(fast_csv_path, nrows=0).columns.tolist()
missing = [v for v in mapping.values() if v and v not in csv_columns]
if missing:
    raise ValueError(f'CSV missing mapped columns: {missing}\nCSV has: {csv_columns}')
csv_row_count = len(fast_input)
if csv_row_count <= 0:
    raise ValueError(f'fast_input.csv is empty (0 rows after spatial filtering)')
print(f'Pre-flight OK: {csv_row_count:,} rows, all mapped columns present')

fast_output_dir = str(WORK_DIR / 'fast_output')
os.makedirs(fast_output_dir, exist_ok=True)

fast_script = str(REPO_DIR / 'FAST-main' / 'Python_env' / 'run_fast.py')
cmd = [
    sys.executable, fast_script,
    '--inventory', fast_csv_path,
    '--mapping-json', mapping_path,
    '--flc', params['flood_load_condition'],
    '--rasters', raster_path,
    '--output-dir', fast_output_dir,
    '--project-root', str(REPO_DIR / 'FAST-main'),
]

print('Running FAST engine...')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=params['fast_timeout'])

# --- Parse FAST result (stdout may contain non-JSON print output) ---
fast_payload = None
for line in reversed(result.stdout.strip().splitlines()):
    try:
        fast_payload = json.loads(line)
        break
    except json.JSONDecodeError:
        continue

if result.returncode != 0:
    error_detail = ''
    if fast_payload:
        error_detail = fast_payload.get('error') or fast_payload.get('message', '')
    if not error_detail:
        error_detail = result.stdout[-2000:] if result.stdout else '(empty stdout)'
    print(f'FAST error detail:\n{error_detail}')
    print(f'FAST STDERR (last 2000 chars):\n{result.stderr[-2000:]}')
    raise RuntimeError(f'FAST failed (rc={result.returncode}): {error_detail}')

# --- Check row-level errors on success ---
row_errors = fast_payload.get('row_errors', 0) if fast_payload else 0
if row_errors > 0:
    raise RuntimeError(f'FAST completed but {row_errors} rows had processing errors — results are incomplete')

print('FAST completed successfully.')

# Find predictions output
pred_files = sorted(Path(fast_output_dir).rglob('*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not pred_files:
    raise FileNotFoundError(f'FAST produced no output CSV in {fast_output_dir}')

predictions_path = str(pred_files[0])
print(f'Predictions file: {predictions_path}')

Pre-flight OK: 4,587,280 rows, all mapped columns present
Running FAST engine...
FAST completed successfully.
Predictions file: /content/repo/outputs/fast_output/fast_input_BERYL_2024_adv29_e10_ResultMaskRaster_sorted.csv


In [ ]:
# Cell 7: Load Predictions + Derive Census Tract GEOID

pred_df = pd.read_csv(predictions_path, dtype={'FltyId': str})
print(f'Loaded {len(pred_df):,} predictions')

if pred_df.empty:
    raise RuntimeError('Predictions file is empty. FAST may have failed silently.')

pred_df.columns = pred_df.columns.str.lower()

# Dedup: MAX(bldgdmgpct) per FltyId
pred_df = (
    pred_df.sort_values('bldgdmgpct', ascending=False)
    .drop_duplicates(subset='fltyid', keep='first')
)
print(f'After dedup: {len(pred_df):,} unique buildings')

# JOIN with NSI to get cbfips (written in Cell 5 alongside FAST input)
_nsi_join = WORK_DIR / 'nsi_cbfips_join.csv'
if not _nsi_join.is_file():
    raise FileNotFoundError(f'Missing {_nsi_join}. Run Cell 5 first to build FAST input + cbfips join table.')
nsi_cbfips = pd.read_csv(_nsi_join, dtype={'fltyid': str, 'cbfips': str})

pred_df = pred_df.merge(nsi_cbfips, on='fltyid', how='left')
matched = pred_df['cbfips'].notna().sum()
total = len(pred_df)
print(f'Census block FIPS matched: {matched:,} / {total:,} ({matched/max(total,1)*100:.1f}%)')

# Derive tract GEOID (first 11 digits of 15-digit cbfips)
pred_df['cbfips'] = pred_df['cbfips'].astype(str).str.zfill(15)
pred_df['tract_geoid'] = pred_df['cbfips'].str[:11]

# Filter residential only
pred_df = pred_df[pred_df['occ'].str.startswith('RES', na=False)].copy()
print(f'Residential with tract GEOID: {len(pred_df):,}, unique tracts: {pred_df["tract_geoid"].nunique():,}')

/tmp/ipykernel_5362/2994068127.py:3: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  pred_df = pd.read_csv(predictions_path, dtype={'FltyId': str})


Loaded 4,587,280 predictions
After dedup: 4,587,280 unique buildings
Census block FIPS matched: 4,587,280 / 4,587,280 (100.0%)
Residential with tract GEOID: 4,101,923, unique tracts: 2,372


In [ ]:
# Cell 8: Classify Damage States + Aggregate to Census Tract

def classify_damage_state(
    dmg_pct: pd.Series, thresholds: dict = None
) -> pd.Series:
    """Classify BldgDmgPct into Hazus damage states."""
    if thresholds is None:
        thresholds = params['DAMAGE_STATE_THRESHOLDS']
    conditions = [
        dmg_pct >= thresholds['Complete'][0],
        dmg_pct >= thresholds['Extensive'][0],
        dmg_pct >= thresholds['Moderate'][0],
        dmg_pct > 0,
    ]
    choices = ['Complete', 'Extensive', 'Moderate', 'Slight']
    return np.select(conditions, choices, default='None')


# Classify each building
pred_df['damage_state'] = classify_damage_state(pred_df['bldgdmgpct'])
state_dist = pred_df['damage_state'].value_counts()
print('Building damage state distribution:')
for ds in ['Slight', 'Moderate', 'Extensive', 'Complete', 'None']:
    n = state_dist.get(ds, 0)
    print(f'  {ds:>10s}: {n:>8,} ({n/len(pred_df)*100:.1f}%)')

# One-hot encode damage states for fast aggregation
for ds in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    pred_df[f'is_{ds}'] = (pred_df['damage_state'] == ds).astype(int)

# Aggregate to census tract
tract_agg = pred_df.groupby('tract_geoid').agg(
    Total_Num_Building=('fltyid', 'count'),
    max_intensity=('bldgdmgpct', 'max'),
    Total_Num_Building_Slight=('is_Slight', 'sum'),
    Total_Num_Building_Moderate=('is_Moderate', 'sum'),
    Total_Num_Building_Extensive=('is_Extensive', 'sum'),
    Total_Num_Building_Complete=('is_Complete', 'sum'),
).reset_index()

tract_agg['max_intensity'] = tract_agg['max_intensity'] / 100.0

# Destruction rates for severity classification
tract_agg['pct_destroyed'] = tract_agg['Total_Num_Building_Complete'] / tract_agg['Total_Num_Building']
tract_agg['pct_major_damage'] = (
    (tract_agg['Total_Num_Building_Extensive'] + tract_agg['Total_Num_Building_Complete'])
    / tract_agg['Total_Num_Building']
)

print(f'\nAggregated to {len(tract_agg):,} census tracts')
tract_agg.head()

Building damage state distribution:
      Slight:    1,069 (0.0%)
    Moderate:    5,505 (0.1%)
   Extensive:    9,185 (0.2%)
    Complete:   64,963 (1.6%)
        None: 4,021,201 (98.0%)

Aggregated to 2,372 census tracts


,tract_geoid,Total_Num_Building,max_intensity,Total_Num_Building_Slight,Total_Num_Building_Moderate,Total_Num_Building_Extensive,Total_Num_Building_Complete,pct_destroyed,pct_major_damage
0,22011960600,498,0.000,0,0,0,0,0.0,0.000000
1,22019003400,228,0.000,0,0,0,0,0.0,0.000000
2,22019003500,1085,0.000,0,0,0,0,0.0,0.000000
3,22019003600,1442,0.495,0,0,1,0,0.0,0.000693
4,22023970201,172,0.000,0,0,0,0,0.0,0.000000


In [ ]:
# Cell 9: Classify Tract Severity (risk_level)

def classify_tract_severity(
    pct_destroyed: pd.Series, pct_major: pd.Series, thresholds: dict = None
) -> pd.Series:
    if thresholds is None:
        thresholds = params['TRACT_SEVERITY']
    conditions = [
        (pct_destroyed >= thresholds['high']['pct_destroyed']) |
        (pct_major >= thresholds['high']['pct_major_damage']),
        (pct_destroyed >= thresholds['medium']['pct_destroyed']) |
        (pct_major >= thresholds['medium']['pct_major_damage']),
    ]
    return np.select(conditions, ['high', 'medium'], default='low')


tract_agg['risk_level'] = classify_tract_severity(
    tract_agg['pct_destroyed'], tract_agg['pct_major_damage']
)

risk_dist = tract_agg['risk_level'].value_counts()
print('Tract severity distribution:')
for level in ['low', 'medium', 'high']:
    n = risk_dist.get(level, 0)
    print(f'  {level:>6s}: {n:>6,} tracts ({n/len(tract_agg)*100:.1f}%)')

Tract severity distribution:
     low:  2,292 tracts (96.6%)
  medium:     37 tracts (1.6%)
    high:     43 tracts (1.8%)


In [ ]:
# Cell 10: Compute BHI Factor (vectorized)

usability = params['BLDNG_USABILITY']
ul_severity = params['UL_SEVERITY']
total_bldg = tract_agg['Total_Num_Building'].replace(0, np.nan)

# Precompute per-tract utility-loss rates (avoids 16x .map(lambda) in the loop)
risk = tract_agg['risk_level']
ul_fu_low  = risk.map({r: ul_severity[r]['FU'][0] for r in ul_severity})
ul_fu_high = risk.map({r: ul_severity[r]['FU'][1] for r in ul_severity})
ul_pu_low  = risk.map({r: ul_severity[r]['PU'][0] for r in ul_severity})
ul_pu_high = risk.map({r: ul_severity[r]['PU'][1] for r in ul_severity})

bhi_low = pd.Series(0.0, index=tract_agg.index)
bhi_high = pd.Series(0.0, index=tract_agg.index)

for ds in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    frac = tract_agg[f'Total_Num_Building_{ds}'] / total_bldg
    u = usability[ds]
    bhi_low  += frac * (u['FU'] * ul_fu_low  + u['PU'] * ul_pu_low  + u['NU'] * 1.0)
    bhi_high += frac * (u['FU'] * ul_fu_high + u['PU'] * ul_pu_high + u['NU'] * 1.0)

tract_agg['BHI_factor_low'] = bhi_low.fillna(0)
tract_agg['BHI_factor_high'] = bhi_high.fillna(0)

print(f'BHI factor range:')
print(f'  Low:  {tract_agg["BHI_factor_low"].min():.6f} - {tract_agg["BHI_factor_low"].max():.6f}')
print(f'  High: {tract_agg["BHI_factor_high"].min():.6f} - {tract_agg["BHI_factor_high"].max():.6f}')

BHI factor range:
  Low:  0.000000 - 0.958780
  High: 0.000000 - 0.972540


In [ ]:
# Cell 11: Load Census Population + SVI (Tract Level)

CENSUS_API_BASE = 'https://api.census.gov/data/2022/acs/acs5'
CDC_SVI_URL = 'https://svi.cdc.gov/Documents/Data/2022/csv/SVI_2022_US.csv'
CDC_SVI_REST_URL = 'https://onemap.cdc.gov/onemapservices/rest/services/SVI/CDC_ATSDR_Social_Vulnerability_Index_2022_USA/FeatureServer/2/query'

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm


def fetch_tract_population(state_fips_list: list[str]) -> pd.DataFrame:
    def _fetch_one(state_fips):
        url = f'{CENSUS_API_BASE}?get=B01001_001E&for=tract:*&in=state:{state_fips}'
        resp = requests.get(url, timeout=params['download_timeout'])
        resp.raise_for_status()
        data = resp.json()
        rows = []
        for row in data[1:]:
            pop, st, county, tract = row
            geoid = f'{st}{county}{tract}'
            rows.append({'tract_geoid': geoid, 'population': int(float(pop or 0))})
        return state_fips, rows, len(resp.content)

    all_rows = []
    failed = []
    t0 = time.time()
    total_bytes = 0

    with tqdm(total=len(state_fips_list), desc="Downloading Census tracts", unit="state") as pbar:
        with ThreadPoolExecutor(max_workers=params['census_max_workers']) as pool:
            futures = {pool.submit(_fetch_one, fips): fips for fips in state_fips_list}
            for future in as_completed(futures):
                state_fips = futures[future]
                try:
                    fips, rows, nbytes = future.result()
                    all_rows.extend(rows)
                    total_bytes += nbytes
                    print(f'  State {fips}: {len(rows)} tracts')
                except Exception as e:
                    failed.append(state_fips)
                    print(f'  State {state_fips}: FAILED ({e})')
                finally:
                    elapsed = max(time.time() - t0, 1e-9)
                    mb_s = total_bytes / elapsed / 1024 / 1024
                    pbar.set_postfix_str(f"{mb_s:.2f} MB/s")
                    pbar.update(1)

    if failed:
        warnings.warn(f'Census API failed for states: {failed}')
    return pd.DataFrame(all_rows)


def _download_svi_via_rest() -> pd.DataFrame:
    rows = []
    offset = 0
    page_size = params['svi_rest_page_size']
    t0 = time.time()
    total_bytes = 0

    with tqdm(desc="Downloading CDC SVI (REST)", unit="page") as pbar:
        while True:
            qparams = {
                'where': '1=1',
                'outFields': 'FIPS,RPL_THEMES',
                'returnGeometry': 'false',
                'resultOffset': offset,
                'resultRecordCount': page_size,
                'f': 'pjson',
            }
            resp = requests.get(CDC_SVI_REST_URL, params=qparams, timeout=params['svi_timeout'])
            resp.raise_for_status()
            total_bytes += len(resp.content)
            payload = resp.json()
            features = payload.get('features', [])
            if not features:
                break

            for feature in features:
                attrs = feature.get('attributes', {})
                rows.append({
                    'FIPS': attrs.get('FIPS'),
                    'RPL_THEMES': attrs.get('RPL_THEMES'),
                })

            offset += len(features)
            pbar.update(1)
            elapsed = max(time.time() - t0, 1e-9)
            mb_s = total_bytes / elapsed / 1024 / 1024
            pbar.set_postfix_str(f"{mb_s:.2f} MB/s")

            if not payload.get('exceededTransferLimit'):
                break

    return pd.DataFrame(rows)


def fetch_tract_svi() -> pd.DataFrame:
    svi_path = WORK_DIR / 'svi_2022_tract.csv'
    if not svi_path.exists():
        print('  Downloading CDC SVI 2022 tract-level data...')
        try:
            resp = requests.get(CDC_SVI_URL, timeout=params['svi_timeout'], stream=True)
            resp.raise_for_status()
            total = int(resp.headers.get('Content-Length', 0) or 0)
            chunk_size = 1024 * 1024
            t_dl = time.time()
            written = 0

            with open(svi_path, 'wb') as f, tqdm(
                total=total if total > 0 else None,
                desc='  CDC SVI CSV',
                unit='B',
                unit_scale=True,
            ) as pbar:
                for chunk in resp.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    f.write(chunk)
                    written += len(chunk)
                    pbar.update(len(chunk))
                    elapsed = max(time.time() - t_dl, 1e-9)
                    mb_s = written / elapsed / 1024 / 1024
                    pbar.set_postfix_str(f"{mb_s:.2f} MB/s")
        except requests.HTTPError:
            print('  CSV URL unavailable, falling back to CDC ArcGIS REST service...')
            _download_svi_via_rest().to_csv(svi_path, index=False)
        print(f'  Saved to {svi_path}')
    else:
        print('  Loading cached SVI data...')

    df = pd.read_csv(svi_path, dtype={'FIPS': str}, usecols=['FIPS', 'RPL_THEMES'])
    df = df.rename(columns={'FIPS': 'tract_geoid', 'RPL_THEMES': 'SVI_Value'})
    df['tract_geoid'] = df['tract_geoid'].str.zfill(11)
    df['SVI_Value'] = pd.to_numeric(df['SVI_Value'], errors='coerce').fillna(0)
    df.loc[df['SVI_Value'] < 0, 'SVI_Value'] = 0  # sentinel -999
    return df


def map_svi_to_shelter_rate(svi_values: pd.Series, thresholds: list) -> pd.Series:
    """Map SVI score to shelter-seeking rate using configurable bin edges."""
    svi_bins = params['SVI_BINS']
    return np.select(
        [svi_values < svi_bins[0], svi_values < svi_bins[1], svi_values >= svi_bins[1]],
        thresholds,
        default=0.0,
    )


# Fetch data
state_fips_in_data = sorted(tract_agg['tract_geoid'].str[:2].unique().tolist())
print(f'Fetching Census population for states: {state_fips_in_data}')
census_df = fetch_tract_population(state_fips_in_data)
print(f'Census data: {len(census_df):,} tracts')

print('\nFetching CDC SVI tract-level data...')
svi_df = fetch_tract_svi()
print(f'SVI data: {len(svi_df):,} tracts')

# Join
tract_agg = tract_agg.merge(census_df, on='tract_geoid', how='left')
tract_agg['population'] = tract_agg['population'].fillna(0).astype(int)

tract_agg = tract_agg.merge(svi_df, on='tract_geoid', how='left')
tract_agg['SVI_Value'] = tract_agg['SVI_Value'].fillna(0)
tract_agg['SVI_Value_Mapped'] = map_svi_to_shelter_rate(
    tract_agg['SVI_Value'], params['SVI_SHELTER_RATES']
)

pop_matched = (tract_agg['population'] > 0).sum()
svi_matched = (tract_agg['SVI_Value'] > 0).sum()
print(f'\nMatched: {pop_matched}/{len(tract_agg)} tracts with Census pop, {svi_matched}/{len(tract_agg)} with SVI')

Fetching Census population for states: ['22', '48']


  State 22: 1388 tracts
  State 48: 6896 tracts
Census data: 8,284 tracts

Fetching CDC SVI tract-level data...
  CSV URL unavailable, falling back to CDC ArcGIS REST service...


  Saved to /content/repo/outputs/svi_2022_tract.csv
SVI data: 84,120 tracts

Matched: 1524/2372 tracts with Census pop, 1523/2372 with SVI


In [ ]:
# Cell 12: Compute Shelter-Seeking Population + Overview

tract_agg['shelter_seeking_low'] = (
    tract_agg['population'] * tract_agg['BHI_factor_low'] * tract_agg['SVI_Value_Mapped']
)
tract_agg['shelter_seeking_high'] = (
    tract_agg['population'] * tract_agg['BHI_factor_high'] * tract_agg['SVI_Value_Mapped']
)

# Build final output
output_cols = [
    'tract_geoid', 'max_intensity', 'population', 'Total_Num_Building',
    'risk_level', 'BHI_factor_low', 'BHI_factor_high',
    'shelter_seeking_low', 'shelter_seeking_high',
    'Total_Num_Building_Slight', 'Total_Num_Building_Moderate',
    'Total_Num_Building_Extensive', 'Total_Num_Building_Complete',
    'SVI_Value', 'SVI_Value_Mapped',
]
final_output = tract_agg[output_cols].rename(columns={'tract_geoid': 'GEOID'})

# Overview
n_tracts = len(final_output)
total_pop = final_output['population'].sum()
total_shelter_low = final_output['shelter_seeking_low'].sum()
total_shelter_high = final_output['shelter_seeking_high'].sum()
avg_intensity = final_output['max_intensity'].mean()

w = 60
print(f'{"":=<{w}}')
print(f'{"SHELTER DEMAND MODEL \u2014 OVERVIEW":^{w}}')
print(f'{"":=<{w}}')
print(f'  Storm: {params["storm_name"]} ({params["storm_id"]}), Advisory #{params["advisory"]}')
print(f'{"":_<{w}}')
print(f'  Average Storm Surge Intensity:      {avg_intensity:>12.4f}')
print(f'  Number of Tracts Impacted:          {n_tracts:>12,}')
print(f'  Total Population Impacted:          {total_pop:>12,}')
print(f'  Shelter-Seeking Population (Low):   {total_shelter_low:>12.1f}')
print(f'  Shelter-Seeking Population (High):  {total_shelter_high:>12.1f}')
print(f'{"":_<{w}}')

for cat in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    col = f'Total_Num_Building_{cat}'
    total = final_output[col].sum()
    avg = final_output[col].mean()
    pct = total / max(final_output['Total_Num_Building'].sum(), 1) * 100
    print(f'  {cat:>10s}: avg={avg:>8.1f}  total={total:>10,.0f}  ({pct:.1f}%)')

print(f'{"":=<{w}}')

for level in ['low', 'medium', 'high']:
    s = final_output[final_output['risk_level'] == level]
    print(f'  {level.upper()} risk: {len(s)} tracts, shelter={s["shelter_seeking_low"].sum():.1f}-{s["shelter_seeking_high"].sum():.1f}')

final_output.head(10)

              SHELTER DEMAND MODEL — OVERVIEW               
  Storm: BERYL (AL022024), Advisory #29
____________________________________________________________
  Average Storm Surge Intensity:            0.2093
  Number of Tracts Impacted:                 2,372
  Total Population Impacted:             6,630,222
  Shelter-Seeking Population (Low):         4752.6
  Shelter-Seeking Population (High):        4889.3
____________________________________________________________
      Slight: avg=     0.5  total=     1,069  (0.0%)
    Moderate: avg=     2.3  total=     5,505  (0.1%)
   Extensive: avg=     3.9  total=     9,185  (0.2%)
    Complete: avg=    27.4  total=    64,963  (1.6%)
  LOW risk: 2292 tracts, shelter=657.4-668.5
  MEDIUM risk: 37 tracts, shelter=541.7-559.7
  HIGH risk: 43 tracts, shelter=3553.6-3661.1


,GEOID,max_intensity,population,Total_Num_Building,risk_level,BHI_factor_low,BHI_factor_high,shelter_seeking_low,shelter_seeking_high,Total_Num_Building_Slight,Total_Num_Building_Moderate,Total_Num_Building_Extensive,Total_Num_Building_Complete,SVI_Value,SVI_Value_Mapped
0,22011960600,0.000,3000,498,low,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.5957,0.025
1,22019003400,0.000,5362,228,low,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.1839,0.000
2,22019003500,0.000,3299,1085,low,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.7713,0.025
3,22019003600,0.495,0,1442,low,0.000191,0.000217,0.000000,0.000000,0,0,1,0,0.0000,0.000
4,22023970201,0.000,0,172,low,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.0000,0.000
5,48007950100,1.000,0,4701,low,0.084519,0.087341,0.000000,0.000000,53,170,46,391,0.0000,0.000
6,48007950200,1.000,1155,1210,low,0.020723,0.020806,0.000000,0.000000,0,0,2,25,0.2067,0.000
7,48007950300,1.000,0,3178,low,0.014485,0.014664,0.000000,0.000000,0,0,14,43,0.0000,0.000
8,48007950400,1.000,3181,1820,low,0.002518,0.005755,0.400465,0.915324,58,59,1,4,0.9360,0.050
9,48007950500,1.000,0,3649,low,0.069928,0.070843,0.000000,0.000000,4,7,68,241,0.0000,0.000


In [ ]:
# Cell 13: Export Results

csv_path = str(WORK_DIR / params['output_csv_name'])
final_output.to_csv(csv_path, index=False)
print(f'CSV exported: {csv_path} ({len(final_output)} rows)')

xlsx_path = str(WORK_DIR / params['output_xlsx_name'])
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    final_output.to_excel(writer, sheet_name='Output', index=False)
    pd.DataFrame({
        'Parameter': [
            'Storm ID', 'Storm Name', 'Advisory #', 'Year',
            'Total Tracts', 'Total Population',
            'Shelter-Seeking (Low)', 'Shelter-Seeking (High)',
        ],
        'Value': [
            params['storm_id'], params['storm_name'], params['advisory'], params['year'],
            n_tracts, total_pop,
            round(total_shelter_low, 1), round(total_shelter_high, 1),
        ],
    }).to_excel(writer, sheet_name='Parameters', index=False)

print(f'Excel exported: {xlsx_path}')

# JSON for pasting back into Excel Step 7
json_output = final_output.to_json(orient='records', double_precision=6)
print(f'\n--- JSON output (first 500 chars) ---')
print(json_output[:500])

try:
    from google.colab import files
    files.download(csv_path)
    files.download(xlsx_path)
except ImportError:
    print(f'\nFiles saved to: {csv_path}, {xlsx_path}')

CSV exported: /content/repo/outputs/shelter_demand_output.csv (2372 rows)
Excel exported: /content/repo/outputs/shelter_demand_output.xlsx

--- JSON output (first 500 chars) ---
[{"GEOID":"22011960600","max_intensity":0.0,"population":3000,"Total_Num_Building":498,"risk_level":"low","BHI_factor_low":0.0,"BHI_factor_high":0.0,"shelter_seeking_low":0.0,"shelter_seeking_high":0.0,"Total_Num_Building_Slight":0,"Total_Num_Building_Moderate":0,"Total_Num_Building_Extensive":0,"Total_Num_Building_Complete":0,"SVI_Value":0.5957,"SVI_Value_Mapped":0.025},{"GEOID":"22019003400","max_intensity":0.0,"population":5362,"Total_Num_Building":228,"risk_level":"low","BHI_factor_low":0.0,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Model Card

**Model**: Shelter Demand Estimator v1.0

**Core Formula**:
```
shelter_seeking = population × BHI_factor × SVI_Value_Mapped

BHI_factor = Σ_d fraction_d × [
    USABILITY[d][FU] × UL_SEVERITY[risk][FU][bound] +
    USABILITY[d][PU] × UL_SEVERITY[risk][PU][bound] +
    USABILITY[d][NU] × 1.0
]
```

**Data Sources**: NHC P-Surge, NSI (USACE), FEMA FAST, Census ACS 5-year, CDC SVI 2022

**Known Limitations**:
1. BldgDmgPct → damage state uses hard thresholds (not fragility curves)
2. Tract population from Census ACS may differ from actual nighttime population
3. SVI mapping uses 3 discrete buckets (0-0.4, 0.4-0.8, 0.8-1.0)